In [ ]:
import os
import re
from pathlib import Path

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from matplotlib.offsetbox import AnnotationBbox, HPacker, TextArea


In [ ]:
def _pdf_subsettable(path):
    """Return True when a font can be subset-embedded in PDF output."""
    try:
        from fontTools.ttLib import TTFont
        from fontTools.subset import Options, Subsetter

        font = TTFont(path)
        subsetter = Subsetter(options=Options())
        subsetter.populate(text='ABCabc0123 .%/')
        subsetter.subset(font)
        return True
    except Exception:
        return False


# Prefer bundled Helvetica; fall back to Nimbus Sans when PDF embedding fails.
FONT_FAMILY = 'DejaVu Sans'
FONT_CANDIDATES = [
    'figutils/Helvetica.ttf',
    '/usr/share/fonts/opentype/urw-base35/NimbusSans-Regular.otf',
]
BOLD_FONT_CANDIDATES = [
    'figutils/Helvetica Bold.ttf',
    '/usr/share/fonts/opentype/urw-base35/NimbusSans-Bold.otf',
]

for candidate in FONT_CANDIDATES:
    if os.path.exists(candidate) and _pdf_subsettable(candidate):
        fm.fontManager.addfont(candidate)
        FONT_FAMILY = fm.FontProperties(fname=candidate).get_name()
        for bold_candidate in BOLD_FONT_CANDIDATES:
            if (
                os.path.exists(bold_candidate)
                and fm.FontProperties(fname=bold_candidate).get_name() == FONT_FAMILY
                and _pdf_subsettable(bold_candidate)
            ):
                fm.fontManager.addfont(bold_candidate)
                break
        plt.rcParams['font.family'] = FONT_FAMILY
        break


ROOT = Path('/home/hsh/hsh/GPT-Disease-Drug-Prediction')
MOE_DIR = ROOT / 'results/0506/xai_attention_moe_kr_val_20260707/report_no_static_anchor/moe'
OUT_DIR = ROOT / 'figure/out'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIST_CSV = MOE_DIR / 'moe_layer_category_expert_distribution_no_static_anchor.csv'
MOE_HEATMAP_PNG = OUT_DIR / 'moe_heatmap.png'
MOE_HEATMAP_PDF = OUT_DIR / 'moe_heatmap.pdf'
MOE_3D_LAYER_AXIS_PNG = OUT_DIR / 'moe_3d_routing.png'
MOE_3D_LAYER_AXIS_PDF = OUT_DIR / 'moe_3d_routing.pdf'
MOE_3D_LAYER_AXIS_CSV = OUT_DIR / 'moe_3d_routing.csv'


# Same chapter palette as figure/auc_scatter.ipynb.
COLOR_REMAP = {
    '#98df8a': '#91c987',
    '#ff7f0e': '#ec713f',
    '#ff9896': '#dd5752',
    '#2ca02c': '#4f7951',
    '#9edae5': '#90c0c2',
    '#7f7f7f': '#87776a',
    '#dbdb8d': '#e6c186',
    '#d62728': '#c34a54',
    '#17becf': '#567797',
    '#8c564b': '#6b512a',
    '#1f77b4': '#2964b8',
    '#ffbb78': '#f7b141',
    '#e377c2': '#e45c88',
    '#FFFF00': '#E7C547',
    '#000a35': '#1d1f20',
}

LABELS_CANDIDATES = [
    ROOT.parent / 'data' / 'labels_chapter.csv',
    ROOT / 'data' / 'labels_chapter.csv',
]
LABELS_PATH = next((p for p in LABELS_CANDIDATES if p.exists()), None)

CHAPTER_COLORS = {}
DRUG_COLOR = '#E7C547'
DEATH_COLOR = '#1d1f20'
if LABELS_PATH is not None:
    labels_df = pd.read_csv(LABELS_PATH)
    labels_df.columns = ['name', 'token_id', 'chapter_full', 'chapter_short', 'color']
    labels_df['color'] = labels_df['color'].replace(COLOR_REMAP)
    CHAPTER_COLORS = (
        labels_df[['chapter_short', 'color']]
        .drop_duplicates('chapter_short')
        .set_index('chapter_short')['color']
        .to_dict()
    )
    if labels_df['chapter_short'].eq('XX. Diabetes drugs').any():
        DRUG_COLOR = labels_df.loc[
            labels_df['chapter_short'].eq('XX. Diabetes drugs'), 'color'
        ].iloc[0]
    DEATH_COLOR = CHAPTER_COLORS.get('Death', DEATH_COLOR)


def color_for_category(category: str) -> str:
    if category == 'Death':
        return DEATH_COLOR
    if category.startswith('Drug:'):
        return DRUG_COLOR
    if category.startswith('ICD:'):
        chapter_short = category[len('ICD:'):].strip()
        return CHAPTER_COLORS.get(chapter_short, '#4C78A8')
    return '#4C78A8'


def chapter_surface_cmap(
    base_color: str,
    light_blend: float = 0.62,
    mid_blend: float = 0.16,
    dark_factor: float = 0.70,
    n: int = 256,
):
    """Tinted light -> base-hue mid -> dark chapter color."""
    from matplotlib.colors import LinearSegmentedColormap

    light = tuple(min(1.0, c + (1.0 - c) * light_blend) for c in mpl.colors.to_rgb(base_color))
    mid = tuple(min(1.0, c + (1.0 - c) * mid_blend) for c in mpl.colors.to_rgb(base_color))
    dark = tuple(max(0.0, c * dark_factor) for c in mpl.colors.to_rgb(base_color))
    return LinearSegmentedColormap.from_list(
        'chapter_surface', [(0.0, light), (0.45, mid), (1.0, dark)], N=n
    )


CHAPTER_CMAP_LIGHT_BLEND = 0.54
CHAPTER_CMAP_MID_BLEND = 0.16
CHAPTER_CMAP_DARK_FACTOR = 0.70
CHAPTER_CMAP_DRUG_DARK_FACTOR = 0.68
CHAPTER_CMAP_DEATH_LIGHT_BLEND = 0.55
CHAPTER_CMAP_DEATH_MID_BLEND = 0.18
CHAPTER_CMAP_DEATH_DARK_FACTOR = 1.55


# Heatmap categories: broad 2D overview across clinically important disease/drug groups.
HEATMAP_CATEGORIES = [
    'ICD: II. Neoplasms',
    'ICD: IV. Metabolic Diseases',
    'ICD: IX. Circulatory Diseases',
    'ICD: X. Respiratory Diseases',
    'ICD: XI. Digestive Diseases',
    'ICD: XIII. Musculoskeletal Diseases',
    'Drug: Metformin',
    'Drug: Insulin',
    'Death',
]

HEATMAP_CATEGORY_LABELS = {
    'ICD: II. Neoplasms': 'II. Neoplasms',
    'ICD: IV. Metabolic Diseases': 'IV. Metabolic Diseases',
    'ICD: IX. Circulatory Diseases': 'IX. Circulatory Diseases',
    'ICD: X. Respiratory Diseases': 'X. Respiratory Diseases',
    'ICD: XI. Digestive Diseases': 'XI. Digestive Diseases',
    'ICD: XIII. Musculoskeletal Diseases': 'XIII. Musculoskeletal Diseases',
    'Drug: Metformin': 'Metformin',
    'Drug: Insulin': 'Insulin',
    'Death': 'Death',
}


# 3D routing surface categories: smaller panel set for readable layer/expert paths.
SURFACE_3D_CATEGORIES = [
    'ICD: II. Neoplasms',
    'ICD: IV. Metabolic Diseases',
    'ICD: IX. Circulatory Diseases',
    'ICD: XIV. Genitourinary Diseases',
    'Drug: Insulin',
    'Death',
]

SURFACE_3D_CATEGORY_LABELS = {
    'ICD: II. Neoplasms': 'II. Neoplasms',
    'ICD: IV. Metabolic Diseases': 'IV. Metabolic Diseases',
    'ICD: IX. Circulatory Diseases': 'IX. Circulatory Diseases',
    'ICD: XIV. Genitourinary Diseases': 'XIV. Genitourinary Diseases',
    'Drug: Insulin': 'Insulin',
    'Death': 'Death',
}

PLOT_CATEGORIES = list(dict.fromkeys(HEATMAP_CATEGORIES + SURFACE_3D_CATEGORIES))

LAYER_GROUPS = [
    ('Early layers
0-1', [0, 1]),
    ('Mid layers
3 and 5', [3, 5]),
    ('Late layers
6-7', [6, 7]),
]


def split_chapter_title(label: str) -> tuple[str, str]:
    """Split 'IV. Metabolic Diseases' into ('IV.', 'Metabolic Diseases')."""
    match = re.match(r'^([IVXLCDM]+\.)\s+(.*)$', label)
    if match:
        return match.group(1), match.group(2)
    return '', label


def set_chapter_split_title(
    ax,
    label: str,
    fontsize: float = 10,
    y: float = 1.02,
    box_alignment: tuple[float, float] = (0.0, 0.0),
):
    """Bold Roman numeral only for ICD chapters; drugs and Death stay fully bold."""
    chapter_code, chapter_name = split_chapter_title(label)
    children = []
    if chapter_code:
        children.append(TextArea(chapter_code, textprops=dict(fontweight='bold', fontsize=fontsize)))
        children.append(TextArea(' ' + chapter_name, textprops=dict(fontweight='normal', fontsize=fontsize)))
    else:
        children.append(TextArea(chapter_name, textprops=dict(fontweight='bold', fontsize=fontsize)))
    box = HPacker(children=children, align='baseline', pad=0, sep=0)
    ax.add_artist(
        AnnotationBbox(
            box,
            (0.0, y),
            xycoords='axes fraction',
            box_alignment=box_alignment,
            frameon=False,
            pad=0.0,
            annotation_clip=False,
        )
    )


def load_moe_distribution(categories: list[str] | None = None) -> pd.DataFrame:
    df = pd.read_csv(DIST_CSV)
    if categories is not None:
        df = df[df['category'].isin(categories)].copy()
    return df


def prepare_stage_matrix(
    df: pd.DataFrame,
    layers: list[int],
    categories: list[str],
) -> pd.DataFrame:
    grouped = (
        df[df['layer'].isin(layers)]
        .groupby(['category', 'expert'], as_index=False)['weighted_fraction']
        .mean()
    )
    return (
        grouped.pivot(index='category', columns='expert', values='weighted_fraction')
        .reindex(index=categories, columns=range(8))
        .fillna(0.0)
    )


def smooth_routing_surface(
    values: np.ndarray,
    upsample: int = 4,
    sigma: float = 0.0,
):
    """Upsample a layer x expert routing surface for visualization."""
    from scipy.interpolate import RectBivariateSpline
    from scipy.ndimage import gaussian_filter

    values = np.clip(np.asarray(values, dtype=float), 0.0, None)
    n_layer, n_expert = values.shape
    layers = np.arange(n_layer, dtype=float)
    experts = np.arange(n_expert, dtype=float)
    working = values.copy()
    if sigma > 0:
        working = np.clip(gaussian_filter(working, sigma=sigma, mode='nearest'), 0.0, None)

    layer_fine = np.linspace(0, n_layer - 1, (n_layer - 1) * upsample + 1)
    expert_fine = np.linspace(0, n_expert - 1, (n_expert - 1) * upsample + 1)
    spline = RectBivariateSpline(layers, experts, working, kx=2, ky=2)
    values_fine = np.clip(spline(layer_fine, expert_fine, grid=True), 0.0, None)
    layer_grid, expert_grid = np.meshgrid(layer_fine, expert_fine, indexing='ij')
    return layer_grid, expert_grid, values_fine, spline


mpl.rcParams.update(
    {
        'font.family': FONT_FAMILY,
        'font.size': 9,
        'axes.titlesize': 11,
        'axes.labelsize': 9,
        'xtick.labelsize': 8,
        'ytick.labelsize': 9,
        'legend.fontsize': 8,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
    }
)

distribution_df = load_moe_distribution(PLOT_CATEGORIES)
distribution_df.head()


## 2D MoE Heatmap

In [ ]:
def make_moe_2d_heatmap(
    df: pd.DataFrame,
    categories: list[str] = HEATMAP_CATEGORIES,
    category_labels: dict[str, str] = HEATMAP_CATEGORY_LABELS,
) -> None:
    fig = plt.figure(figsize=(16, 5), constrained_layout=False)
    gs = fig.add_gridspec(
        1,
        3,
        width_ratios=[1.0, 1.0, 1.0],
        left=0.105,
        right=0.965,
        top=0.80,
        bottom=0.16,
        wspace=0.08,
    )
    axes = [fig.add_subplot(gs[0, idx]) for idx in range(3)]

    vmin = -0.3
    vmax = 0.65
    row_labels = [category_labels[c] for c in categories]

    for panel_idx, (ax, (title, layers)) in enumerate(zip(axes, LAYER_GROUPS)):
        matrix = prepare_stage_matrix(df, layers, categories)
        values = matrix.to_numpy()

        ax.imshow(
            values,
            aspect='equal',
            cmap='RdYlGn_r',
            vmin=vmin,
            vmax=vmax,
            alpha=0.95,
            interpolation='nearest',
        )
        ax.set_box_aspect(values.shape[0] / values.shape[1])
        ax.set_title(title.split('
')[0].strip(), fontweight='bold', fontsize=11, pad=8)
        ax.set_xlabel('Expert', fontsize=10)
        ax.set_xticks(range(values.shape[1]))
        ax.set_yticks(range(values.shape[0]))
        ax.set_xticklabels(labels=[str(i) for i in range(8)], fontsize=10)
        if panel_idx == 0:
            ax.set_yticklabels(labels=row_labels, fontsize=10)
        else:
            ax.set_yticklabels(labels=[])
        ax.tick_params(axis='both', length=0)

        for row in range(values.shape[0]):
            for col in range(values.shape[1]):
                value = values[row, col]
                if value >= 0.28:
                    text_color = 'white' if value > 0.35 else 'black'
                    ax.text(
                        col,
                        row,
                        f'{value:.2f}',
                        ha='center',
                        va='center',
                        fontsize=9,
                        color=text_color,
                    )

        for row, category in enumerate(categories):
            ax.add_patch(
                plt.Rectangle(
                    (-0.5, row - 0.5),
                    values.shape[1],
                    1,
                    fill=False,
                    edgecolor=color_for_category(category),
                    linewidth=1.5,
                    alpha=0.75,
                )
            )
        ax.set_xticks(np.arange(-0.5, values.shape[1], 1), minor=True)
        ax.set_yticks(np.arange(-0.5, values.shape[0], 1), minor=True)
        ax.grid(which='minor', color='white', linewidth=1.1)
        ax.tick_params(which='minor', bottom=False, left=False)
        for spine in ax.spines.values():
            spine.set_visible(False)

    fig.savefig(MOE_HEATMAP_PNG, dpi=900, bbox_inches='tight', pad_inches=0.03)
    fig.savefig(MOE_HEATMAP_PDF, bbox_inches='tight', pad_inches=0.03)
    plt.close(fig)


make_moe_2d_heatmap(distribution_df)
print(f'Saved: {MOE_HEATMAP_PNG}')
print(f'Saved: {MOE_HEATMAP_PDF}')


In [ ]:
display(Image(filename=str(MOE_HEATMAP_PNG)))

## 3D Layer-Axis Routing Surface

In [ ]:
# Per-panel ICD-10 chapter colormap for 3D routing surfaces.
PANEL_3D_TITLE_Y = 0.95
PANEL_3D_LABELPAD = {'x': -5, 'y': -5}
PANEL_3D_TICKPAD = {'x': -3, 'y': -3, 'z': -1}
ROUTING_SURFACE_ZLIM = 0.8
ROUTING_SURFACE_NORM_VMIN = 0.0
ROUTING_SURFACE_NORM_VMAX = 0.62
ROUTING_SURFACE_GAMMA = 0.88


def make_moe_3d_layer_axis_surface(
    df: pd.DataFrame,
    categories: list[str] = SURFACE_3D_CATEGORIES,
    category_labels: dict[str, str] = SURFACE_3D_CATEGORY_LABELS,
) -> pd.DataFrame:
    fig = plt.figure(figsize=(12.8, 7.4), constrained_layout=False)
    gs = fig.add_gridspec(
        2,
        3,
        left=0.03,
        right=0.875,
        top=0.96,
        bottom=0.03,
        wspace=0.01,
        hspace=0.015,
    )

    top_path_rows = []
    layers = np.arange(8)
    experts = np.arange(8)
    routing_norm = mpl.colors.PowerNorm(
        gamma=ROUTING_SURFACE_GAMMA,
        vmin=ROUTING_SURFACE_NORM_VMIN,
        vmax=ROUTING_SURFACE_NORM_VMAX,
    )

    for panel_idx, category in enumerate(categories, start=1):
        row_idx_panel = (panel_idx - 1) // 3
        col_idx_panel = (panel_idx - 1) % 3
        ax = fig.add_subplot(gs[row_idx_panel, col_idx_panel], projection='3d')
        ax.set_facecolor('none')
        ax.patch.set_alpha(0.0)

        matrix = (
            df[df['category'].eq(category)]
            .pivot(index='layer', columns='expert', values='weighted_fraction')
            .reindex(index=layers, columns=experts)
            .fillna(0.0)
        )
        values = matrix.to_numpy(dtype=float)
        layer_grid, expert_grid, values_plot, surface_spline = smooth_routing_surface(values)

        if category == 'Death':
            panel_light_blend = CHAPTER_CMAP_DEATH_LIGHT_BLEND
            panel_mid_blend = CHAPTER_CMAP_DEATH_MID_BLEND
            panel_dark_factor = CHAPTER_CMAP_DEATH_DARK_FACTOR
        elif category.startswith('Drug:'):
            panel_light_blend = CHAPTER_CMAP_LIGHT_BLEND
            panel_mid_blend = CHAPTER_CMAP_MID_BLEND
            panel_dark_factor = CHAPTER_CMAP_DRUG_DARK_FACTOR
        else:
            panel_light_blend = CHAPTER_CMAP_LIGHT_BLEND
            panel_mid_blend = CHAPTER_CMAP_MID_BLEND
            panel_dark_factor = CHAPTER_CMAP_DARK_FACTOR
        panel_cmap = chapter_surface_cmap(
            color_for_category(category),
            light_blend=panel_light_blend,
            mid_blend=panel_mid_blend,
            dark_factor=panel_dark_factor,
        )

        ax.plot_surface(
            layer_grid,
            expert_grid,
            values_plot,
            cmap=panel_cmap,
            norm=routing_norm,
            linewidth=0.0,
            antialiased=True,
            shade=False,
        )

        ranked_experts = np.argsort(values, axis=1)[:, ::-1]
        first_experts = ranked_experts[:, 0]
        second_experts = ranked_experts[:, 1]
        row_idx = np.arange(len(layers))
        first_values = values[row_idx, first_experts]
        second_values = values[row_idx, second_experts]
        first_path_z = np.clip(
            surface_spline(layers.astype(float), first_experts.astype(float), grid=False),
            0.0,
            None,
        ) + 0.012
        second_path_z = np.clip(
            surface_spline(layers.astype(float), second_experts.astype(float), grid=False),
            0.0,
            None,
        ) + 0.016

        ax.plot(
            layers,
            first_experts,
            first_path_z,
            color='#111111',
            linewidth=1.2,
            marker='o',
            markersize=2.5,
            markerfacecolor='white',
            markeredgecolor='#111111',
            zorder=6,
        )
        ax.plot(
            layers,
            second_experts,
            second_path_z,
            color='#686868',
            linewidth=1.0,
            linestyle='--',
            marker='o',
            markersize=2.5,
            markerfacecolor='#F2F2F2',
            markeredgecolor='#686868',
            alpha=0.88,
            zorder=5,
        )

        for layer, first_expert, first_value, second_expert, second_value in zip(
            layers, first_experts, first_values, second_experts, second_values
        ):
            top_path_rows.append(
                {
                    'category': category,
                    'category_label': category_labels[category],
                    'layer': int(layer),
                    'first_expert': int(first_expert),
                    'first_weighted_fraction': float(first_value),
                    'second_expert': int(second_expert),
                    'second_weighted_fraction': float(second_value),
                    'first_second_margin': float(first_value - second_value),
                }
            )

        stage_z_offsets = (0.025, 0.035, 0.045)
        stage_labels = ('Early', 'Mid', 'Late')
        for stage_label, (_stage_title, group_layers), z_offset in zip(
            stage_labels, LAYER_GROUPS, stage_z_offsets
        ):
            anchor_layer = max(group_layers)
            first_expert = int(first_experts[anchor_layer])
            second_expert = int(second_experts[anchor_layer])
            first_value = float(first_values[anchor_layer])
            if first_value < 0.18:
                continue
            label_color = 'white' if category == 'Death' else '#111111'
            label_outline = '#111111' if category == 'Death' else 'white'
            text_obj = ax.text(
                anchor_layer,
                first_expert,
                first_path_z[anchor_layer] + z_offset,
                f'{stage_label} E{first_expert}/E{second_expert}',
                ha='center',
                va='bottom',
                fontsize=7,
                color=label_color,
                zorder=10,
            )
            text_obj.set_path_effects([
                pe.withStroke(linewidth=1.1, foreground=label_outline, alpha=0.85)
            ])

        set_chapter_split_title(
            ax,
            category_labels[category],
            fontsize=10,
            y=PANEL_3D_TITLE_Y,
            box_alignment=(0.0, 1.0),
        )
        ax.set_xlim(-0.5, 7.5)
        ax.set_ylim(-0.5, 7.5)
        ax.set_zlim(0, ROUTING_SURFACE_ZLIM)
        ax.set_zticks([0.25, 0.5, 0.75])
        ax.tick_params(axis='x', pad=PANEL_3D_TICKPAD['x'])
        ax.tick_params(axis='y', pad=PANEL_3D_TICKPAD['y'])
        ax.tick_params(axis='z', pad=PANEL_3D_TICKPAD['z'])
        ax.set_xlabel('Layer', labelpad=PANEL_3D_LABELPAD['x'])
        ax.set_ylabel('Expert', labelpad=PANEL_3D_LABELPAD['y'])
        ax.set_zlabel('')
        ax.view_init(**VIEW_3D_LAYER_SURFACE)
        ax.set_box_aspect((1.0, 1.0, 0.7))

        for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            axis.pane.fill = False
            axis.pane.set_edgecolor('none')
            axis._axinfo['grid'].update(
                color='#bbbbbb',
                linestyle='--',
                linewidth=0.3,
            )
        ax.grid(True)

    fig.savefig(MOE_3D_LAYER_AXIS_PNG, dpi=900, bbox_inches='tight', pad_inches=0.04)
    fig.savefig(MOE_3D_LAYER_AXIS_PDF, bbox_inches='tight', pad_inches=0.04)
    plt.close(fig)

    top_paths = pd.DataFrame(top_path_rows)
    top_paths.to_csv(MOE_3D_LAYER_AXIS_CSV, index=False)
    return top_paths


In [ ]:
# 3D camera angles (matplotlib Axes3D.view_init):
#   elev = elevation above the x-y plane (degrees, higher = more top-down)
#   azim = rotation around z axis (degrees, negative = view from front-right)
VIEW_3D_LAYER_SURFACE = {'elev': 50, 'azim': -50}

layer_axis_top_paths = make_moe_3d_layer_axis_surface(distribution_df)
print(f'Saved: {MOE_3D_LAYER_AXIS_PNG}')
print(f'Saved: {MOE_3D_LAYER_AXIS_PDF}')
print(f'Saved: {MOE_3D_LAYER_AXIS_CSV}')
display(Image(filename=str(MOE_3D_LAYER_AXIS_PNG)))
